# Comprehensive EDA Pipeline
## Automated Data Profiling & Report Generation

**Supported Formats:** CSV, Excel, JSON, Parquet, Feather, Pickle, TSV

This notebook automatically generates:
- YData Profiling HTML Report
- Sweetviz HTML Report
- AutoViz Visualizations
- D-Tale Interactive Dashboard (in-notebook)

---

## 1. Import Libraries & Configuration

In [4]:
import pandas as pd
import numpy as np
import os
import json
import sys
import warnings
import logging
from pathlib import Path
from datetime import datetime
import chardet
import traceback


warnings.filterwarnings('ignore')

# CONFIGURATION - MODIFY THIS
DATA_FILE = 'datasets/cleaned_salary_survey.csv'  # ← CHANGE THIS TO YOUR FILE
OUTPUT_DIR = 'reports'
CORRELATION_THRESHOLD = 0.5

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✓ Libraries loaded')

✓ Libraries loaded


## 2. Helper Functions

In [7]:

try:
    from charset_normalizer import from_path
    HAS_CHARSET = True
except ImportError:
    HAS_CHARSET = False


def detect_encoding(file_path):
    """
    Detect text file encoding.
    """
    if HAS_CHARSET:
        try:
            result = from_path(file_path).best()
            if result:
                return result.encoding
        except Exception:
            pass

    # Common fallbacks
    return "utf-8"


def load_data(file_path):
    """
    Universal dataset loader.

    Supported formats:
        CSV
        TSV
        TXT
        Excel (.xls/.xlsx)
        JSON
        Parquet
        Feather
        Pickle
        HDF5
        Stata (.dta)
        SPSS (.sav)
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    ext = Path(file_path).suffix.lower()

    # ---------- CSV / TXT ----------
    if ext in [".csv", ".txt", ".tsv"]:

        encoding = detect_encoding(file_path)

        separators = [",", ";", "\t", "|"]

        for sep in separators:
            try:
                df = pd.read_csv(
                    file_path,
                    sep=sep,
                    encoding=encoding,
                    engine="python",
                    on_bad_lines="skip",
                )

                if df.shape[1] > 1:
                    return df

            except Exception:
                pass

        # Last resort
        for enc in ["utf-8", "latin1", "cp1252"]:
            try:
                return pd.read_csv(
                    file_path,
                    encoding=enc,
                    engine="python",
                    on_bad_lines="skip",
                )
            except Exception:
                pass

        raise ValueError("Could not read CSV/TXT file.")

    # ---------- Excel ----------
    elif ext in [".xls", ".xlsx"]:
        return pd.read_excel(file_path)

    # ---------- JSON ----------
    elif ext == ".json":
        try:
            return pd.read_json(file_path)
        except ValueError:
            return pd.read_json(file_path, lines=True)

    # ---------- Parquet ----------
    elif ext == ".parquet":
        return pd.read_parquet(file_path)

    # ---------- Feather ----------
    elif ext == ".feather":
        return pd.read_feather(file_path)

    # ---------- Pickle ----------
    elif ext in [".pkl", ".pickle"]:
        return pd.read_pickle(file_path)

    # ---------- HDF5 ----------
    elif ext in [".h5", ".hdf5"]:
        return pd.read_hdf(file_path)

    # ---------- Stata ----------
    elif ext == ".dta":
        return pd.read_stata(file_path)

    # ---------- SPSS ----------
    elif ext == ".sav":
        return pd.read_spss(file_path)

    else:
        raise ValueError(f"Unsupported file format: {ext}")

## 3. Load Dataset

In [8]:
from datetime import datetime
from pathlib import Path
import os
import sys

start_time = datetime.now()

try:
    df = load_data(DATA_FILE)

    dataset_name = Path(DATA_FILE).stem
    dataset_output_dir = Path(OUTPUT_DIR) / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)

    print("✓ Data loaded successfully!")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Memory: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
    print(df.head())

except Exception as e:
    print(f"✗ Failed to load: {e}")
    sys.exit(1)

✓ Data loaded successfully!
Rows: 28,063
Columns: 22
Memory: 29.38 MB
             timestamp    age                       industry  \
0  2021-04-27 11:02:10  25-34   Education (Higher Education)   
1  2021-04-27 11:02:22  25-34              Computing or Tech   
2  2021-04-27 11:02:38  25-34  Accounting, Banking & Finance   
3  2021-04-27 11:02:41  25-34                     Nonprofits   
4  2021-04-27 11:02:42  25-34  Accounting, Banking & Finance   

                                  job_title job_title_context  annual_salary  \
0        Research and Instruction Librarian           Unknown          55000   
1  Change & Internal Communications Manager           Unknown          54600   
2                      Marketing Specialist           Unknown          34000   
3                           Program Manager           Unknown          62000   
4                        Accounting Manager           Unknown          60000   

   additional_compensation currency         country           st

## 4. Dataset Overview

In [9]:
print('\n' + '='*80)
print('DATASET OVERVIEW')
print('='*80)
print(f'\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\nColumn Information:')
print(df.dtypes)

print('\nFirst 5 rows:')
df.head()


DATASET OVERVIEW

Shape: 28,063 rows × 22 columns
Memory Usage: 29.38 MB

Column Information:
timestamp                   object
age                         object
industry                    object
job_title                   object
job_title_context           object
annual_salary                int64
additional_compensation    float64
currency                    object
country                     object
state                       object
city                        object
total_experience_years      object
field_experience_years      object
education_level             object
gender                      object
race                        object
race_clean                  object
country_clean               object
month                       object
day                          int64
year                         int64
time                        object
dtype: object

First 5 rows:


,timestamp,age,industry,job_title,job_title_context,annual_salary,additional_compensation,currency,country,state,city,total_experience_years,field_experience_years,education_level,gender,race,race_clean,country_clean,month,day,year,time
0,2021-04-27 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,Unknown,55000,0.0,USD,US,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White,White,Us,April,27,2021,11:02:10
1,2021-04-27 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,Unknown,54600,4000.0,GBP,United Kingdom,Unknown,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White,White,United Kingdom,April,27,2021,11:02:22
2,2021-04-27 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,Unknown,34000,0.0,USD,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White,White,Us,April,27,2021,11:02:38
3,2021-04-27 11:02:41,25-34,Nonprofits,Program Manager,Unknown,62000,3000.0,USD,US,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White,White,Us,April,27,2021,11:02:41
4,2021-04-27 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,Unknown,60000,7000.0,USD,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White,White,Us,April,27,2021,11:02:42


## 5. Data Types & Missing Values

In [10]:
print('\n' + '='*80)
print('DATA TYPES & MISSING VALUES')
print('='*80)

info_df = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.values,
    'Non-Null': df.count().values,
    'Null': df.isnull().sum().values,
    'Null %': (df.isnull().sum().values / len(df) * 100).round(2)
})

print('\n', info_df.to_string(index=False))
print(f'\nTotal Missing Values: {df.isnull().sum().sum()}')


DATA TYPES & MISSING VALUES

                  Column    Type  Non-Null  Null  Null %
              timestamp  object     28063     0     0.0
                    age  object     28063     0     0.0
               industry  object     28063     0     0.0
              job_title  object     28063     0     0.0
      job_title_context  object     28063     0     0.0
          annual_salary   int64     28063     0     0.0
additional_compensation float64     28063     0     0.0
               currency  object     28063     0     0.0
                country  object     28063     0     0.0
                  state  object     28063     0     0.0
                   city  object     28063     0     0.0
 total_experience_years  object     28063     0     0.0
 field_experience_years  object     28063     0     0.0
        education_level  object     28063     0     0.0
                 gender  object     28063     0     0.0
                   race  object     28063     0     0.0
             race

## 6. Numerical Statistics

In [11]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numerical_cols:
    print('\n' + '='*80)
    print('NUMERICAL STATISTICS')
    print('='*80)
    stats = df[numerical_cols].describe(include='all').T
    stats['skew'] = df[numerical_cols].skew()
    stats['kurtosis'] = df[numerical_cols].kurtosis()
    print('\n', stats.round(4))
else:
    print('\nNo numerical columns found.')


NUMERICAL STATISTICS

                            count         mean           std     min      25%  \
annual_salary            28063.0  372763.9462  3.623781e+07     0.0  54000.0   
additional_compensation  28063.0   13399.6176  7.175214e+05     0.0      0.0   
day                      28063.0      24.3307  8.060400e+00     1.0     27.0   
year                     28063.0    2021.0456  3.602000e-01  2021.0   2021.0   

                             50%       75%           max      skew    kurtosis  
annual_salary            75000.0  109200.0  6.000070e+09  162.2409  26790.9190  
additional_compensation      0.0    5000.0  1.200000e+08  166.6542  27868.8621  
day                         27.0      28.0  3.100000e+01   -1.9987      2.2946  
year                      2021.0    2021.0  2.026000e+03    9.6096    101.1628  


## 7. Categorical Statistics

In [12]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_cols:
    print('\n' + '='*80)
    print('CATEGORICAL STATISTICS')
    print('='*80)
    for col in categorical_cols:
        print(f'\n{col}:')
        print(f'  Unique: {df[col].nunique()}')
        print(f'  Top 5 values:')
        print(df[col].value_counts().head().to_string())
else:
    print('\nNo categorical columns found.')


CATEGORICAL STATISTICS

timestamp:
  Unique: 25293
  Top 5 values:
timestamp
2021-04-27 12:05:06    5
2021-04-27 11:24:33    5
2021-04-27 11:05:17    5
2021-04-27 11:30:18    5
2021-04-27 11:56:18    5

age:
  Unique: 7
  Top 5 values:
age
25-34    12625
35-44     9870
45-54     3181
18-24     1284
55-64      990

industry:
  Unique: 1225
  Top 5 values:
industry
Computing or Tech                       4698
Education (Higher Education)            2474
Nonprofits                              2405
Health care                             1894
Government and Public Administration    1886

job_title:
  Unique: 14433
  Top 5 values:
job_title
Software Engineer           286
Project Manager             229
Senior Software Engineer    196
Director                    195
Program Manager             153

job_title_context:
  Unique: 7029
  Top 5 values:
job_title_context
Unknown           20812
Fundraising          20
Attorney              8
Public library        7
Librarian             7

curr

## 8. Correlation Matrix

In [13]:
if len(numerical_cols) > 1:
    print('\n' + '='*80)
    print('CORRELATION MATRIX')
    print('='*80)
    corr_matrix = df[numerical_cols].corr()
    print('\nFull Correlation Matrix:')
    print(corr_matrix.round(3))
    
    print(f'\nHigh Correlations (abs > {CORRELATION_THRESHOLD}):')
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > CORRELATION_THRESHOLD:
                high_corr.append({
                    'Column 1': corr_matrix.columns[i],
                    'Column 2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    if high_corr:
        print(pd.DataFrame(high_corr).to_string(index=False))
    else:
        print(f'No correlations above {CORRELATION_THRESHOLD} threshold.')
else:
    print('\nNot enough numerical columns for correlation analysis.')


CORRELATION MATRIX

Full Correlation Matrix:
                         annual_salary  additional_compensation    day   year
annual_salary                    1.000                    0.143 -0.010  0.053
additional_compensation          0.143                    1.000  0.002 -0.001
day                             -0.010                    0.002  1.000 -0.151
year                             0.053                   -0.001 -0.151  1.000

High Correlations (abs > 0.5):
No correlations above 0.5 threshold.


## 9. Generate panda profiling

In [14]:
print('\n' + '='*80)
print('DATA PROFILING REPORT')
print('='*80)

try:
    print('\nGenerating data profile summary...')
    
    profile_file = os.path.join(dataset_output_dir, 'data_profile_report.html')
    
    html_content = f'''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Data Profile Report - {dataset_name}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
            .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 20px; border-radius: 8px; }}
            h1 {{ color: #333; border-bottom: 3px solid #007bff; padding-bottom: 10px; }}
            h2 {{ color: #555; margin-top: 30px; }}
            table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
            th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
            th {{ background-color: #007bff; color: white; }}
            tr:hover {{ background-color: #f9f9f9; }}
            .stat-box {{ display: inline-block; margin: 10px; padding: 15px; background: #f9f9f9; border-left: 4px solid #007bff; }}
            .section {{ margin: 20px 0; padding: 15px; background: #f9f9f9; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>📊 Data Profile Report: {dataset_name}</h1>
            
            <div class="section">
                <h2>📈 Dataset Overview</h2>
                <div class="stat-box">
                    <strong>Rows:</strong> {df.shape[0]:,}
                </div>
                <div class="stat-box">
                    <strong>Columns:</strong> {df.shape[1]}
                </div>
                <div class="stat-box">
                    <strong>Memory:</strong> {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB
                </div>
                <div class="stat-box">
                    <strong>Missing Values:</strong> {df.isnull().sum().sum()}
                </div>
            </div>
            
            <h2>📋 Column Information</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Type</th>
                    <th>Non-Null</th>
                    <th>Null</th>
                    <th>Null %</th>
                </tr>
    '''
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].dtype}</td>
                    <td>{df[col].count():,}</td>
                    <td>{null_count}</td>
                    <td>{null_pct:.2f}%</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <h2>🔢 Numerical Statistics</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Mean</th>
                    <th>Std</th>
                    <th>Min</th>
                    <th>Max</th>
                </tr>
    '''
    
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numerical_cols:
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].mean():.4f}</td>
                    <td>{df[col].std():.4f}</td>
                    <td>{df[col].min():.4f}</td>
                    <td>{df[col].max():.4f}</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <h2>📊 Top Values (Categorical)</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Unique</th>
                    <th>Top Value</th>
                    <th>Frequency</th>
                </tr>
    '''
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in categorical_cols:
        top_val = df[col].value_counts().index[0] if len(df[col].value_counts()) > 0 else 'N/A'
        top_count = df[col].value_counts().values[0] if len(df[col].value_counts()) > 0 else 0
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].nunique()}</td>
                    <td>{top_val}</td>
                    <td>{top_count}</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <footer style="margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; text-align: center; color: #666;">
                <p>Generated automatically using Python Pandas</p>
            </footer>
        </div>
    </body>
    </html>
    '''
    
    with open(profile_file, 'w') as f:
        f.write(html_content)
    
    print(f'✓ Custom profiling report saved: {profile_file}')
    if os.path.exists(profile_file):
        file_size_mb = os.path.getsize(profile_file) / 1024**2
        print(f'  File size: {file_size_mb:.2f} MB')
        print(f'  Open in browser: {profile_file}')

except Exception as e:
    print(f'⚠ Error: {str(e)[:200]}')
    import traceback
    print(traceback.format_exc()[:300])


DATA PROFILING REPORT

Generating data profile summary...
✓ Custom profiling report saved: reports/cleaned_salary_survey/data_profile_report.html
  File size: 0.01 MB
  Open in browser: reports/cleaned_salary_survey/data_profile_report.html


## 10. Generate Sweetviz Report

In [15]:
import os
import sweetviz as sv
import warnings

warnings.filterwarnings("ignore")

print("\n" + "=" * 80)
print("SWEETVIZ REPORT")
print("=" * 80)

try:
    print("\nGenerating Sweetviz report...")

    # Create report
    my_report = sv.analyze(df)

    # Output path
    sweetviz_file = os.path.join(dataset_output_dir, "sweetviz_report.html")

    # Save HTML report (correct method)
    my_report.show_html(
        filepath=sweetviz_file,
        open_browser=False,
        layout="widescreen",
        scale=1.0
    )

    print(f"✓ Sweetviz report saved: {sweetviz_file}")

    # File validation
    if os.path.exists(sweetviz_file):
        size_mb = os.path.getsize(sweetviz_file) / (1024 ** 2)
        print(f"  File size: {size_mb:.2f} MB")
    else:
        print("⚠ File was not created properly")

except ImportError:
    print("⚠ Sweetviz not installed.")
    print("Install using: pip install sweetviz")

except Exception as e:
    print(f"⚠ Error generating Sweetviz report:\n{e}")

    # Safe fallback (FIXED)
    try:
        print("\nTrying fallback method...")

        sweetviz_file = os.path.join(dataset_output_dir, "sweetviz_report_fallback.html")

        my_report.show_html(
            filepath=sweetviz_file,
            open_browser=False
        )

        print(f"✓ Fallback report saved: {sweetviz_file}")

    except Exception as e2:
        print(f"❌ Fallback also failed: {e2}")


SWEETVIZ REPORT

Generating Sweetviz report...


                                             |          | [  0%]   00:00 -> (? left)

Report reports/cleaned_salary_survey/sweetviz_report.html was generated.
✓ Sweetviz report saved: reports/cleaned_salary_survey/sweetviz_report.html
  File size: 1.45 MB


## 11. Generate AutoViz Visualizations

In [16]:
import os
from pathlib import Path
import warnings
from autoviz.AutoViz_Class import AutoViz_Class

warnings.filterwarnings("ignore")

print("\n" + "=" * 80)
print("AUTOVIZ VISUALIZATIONS")
print("=" * 80)

try:
    print("\nGenerating AutoViz plots...")

    autoviz_dir = os.path.join(dataset_output_dir, "autoviz_plots")
    Path(autoviz_dir).mkdir(parents=True, exist_ok=True)

    AV = AutoViz_Class()

    dft = AV.AutoViz(
        filename=DATA_FILE,
        sep=",",
        depVar="",
        dfte=None,
        header=0,
        verbose=1,
        lowess=False,
        chart_format="png",
        max_rows_analyzed=len(df),
        max_cols_analyzed=len(df.columns),
        save_plot_dir=autoviz_dir
    )

    print("\n✓ AutoViz execution completed")

except ImportError:
    print("⚠ AutoViz is not installed.")

except Exception as e:
    print(f"⚠ Error during AutoViz execution:\n{e}")

Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)

AUTOVIZ VISUALIZATIONS

Generating AutoViz plots...
Shape of your Data Set loaded: (28063, 22)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  1
    Number of Integer-Categorical Columns =  2
    Number of String-Categorical Columns =  10
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  0
    Number of Numeric-Boolean Columns =  0
    Number of Discrete String Columns =

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
timestamp,object,0.000000,90,,,25293 rare categories: Too many to list. Group them into a single category or drop the categories.
age,object,0.000000,0,,,"2 rare categories: ['65 or over', 'under 18']. Group them into a single category or drop the categories."
industry,object,0.000000,4,,,No issue
job_title,object,0.000000,51,,,No issue
job_title_context,object,0.000000,25,,,No issue
annual_salary,int64,0.000000,13,0.000000,6000070000.000000,Column has 1215 outliers greater than upper bound (192000.00) or lower than lower bound(-28800.00). Cap them or remove them.
additional_compensation,float64,0.000000,NA,0.000000,120000000.000000,Column has 4003 outliers greater than upper bound (12500.00) or lower than lower bound(-7500.00). Cap them or remove them.
currency,object,0.000000,0,,,"6 rare categories: ['Other', 'SEK', 'CHF', 'JPY', 'ZAR', 'HKD']. Group them into a single category or drop the categories."
country,object,0.000000,1,,,Possible high cardinality column with 379 unique values: Use hash encoding or text embedding to reduce dimension.
state,object,0.000000,0,,,Possible high cardinality column with 139 unique values: Use hash encoding or text embedding to reduce dimension.


[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /home/abubakar/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /home/abubakar/nltk_data...
[nltk_data]    |   Unzipping corpora/gazetteers.zip.
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     /home/abubakar/nltk_data...
[nltk_data]    |   Unzipping corpora/genesis.zip.
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     /home/abubakar/nltk_data...
[nltk_data]    |   Unzipping corpora/gutenberg.zip.
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     /home/abubakar/nltk_data...
[nltk_data]    |   Unzipping corpora/inaugural.zip.
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /home/abubakar/nltk_data...
[nltk_data]    |   Unzipping corpora/movie_reviews.zip.
[nltk_data]    | Downloading 

Could not draw wordcloud plot for industry. 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk.org/data.html
If this doesn't fix the problem, file an issue at https://github.com/sloria/TextBlob/issues.

Could not draw wordcloud plot for job_title. 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk.org/data.html
If this doesn't fix the problem, file an issue at https://github.com/sloria/TextBlob/issues.

Could not draw wordcloud plot for job_title_context. 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing 

In [17]:
import os
import matplotlib.pyplot as plt

os.makedirs(autoviz_dir, exist_ok=True)

fig_nums = plt.get_fignums()

print(f"Total figures found: {len(fig_nums)}")

for i, fig_num in enumerate(fig_nums):
    fig = plt.figure(fig_num)

    # FORCE single chart isolation
    fig.canvas.draw()

    save_path = os.path.join(autoviz_dir, f"autoviz_chart_{i+1}.png")

    fig.savefig(save_path, bbox_inches="tight", dpi=300)

print(f"✓ Saved {len(fig_nums)} individual charts to: {autoviz_dir}")

Total figures found: 14
✓ Saved 14 individual charts to: reports/cleaned_salary_survey/autoviz_plots


## 12. Launch D-Tale Interactive Dashboard

In [18]:

print('\n' + '='*80)
print('D-TALE INTERACTIVE DASHBOARD')
print('='*80)

try:
    import dtale
    import warnings
    warnings.filterwarnings('ignore')
    print('\nLaunching D-Tale dashboard...')
    
    d = dtale.show(df, open_browser=False)
    
    print(f'✓ D-Tale is running!')
    print("D-Tale URL:", d._main_url)
    print('\nD-Tale Features:')
    print('  - Interactive data exploration')
    print('  - Column statistics')
    print('  - Filtering & sorting')
    print('  - Visualization tools')
    print('\nNote: Keep this notebook running to access D-Tale.')
    print('\nTo stop D-Tale, uncomment and run the cleanup cell below.')
    
    dtale_instance = d

except ImportError:
    print('⚠ D-Tale not installed.')
    print('  Install: pip install dtale')
except Exception as e:
    print(f'⚠ Error launching D-Tale: {str(e)[:200]}')
    print('  Try updating: pip install --upgrade dtale')



D-TALE INTERACTIVE DASHBOARD

Launching D-Tale dashboard...
✓ D-Tale is running!
D-Tale URL: http://phoenix:40000/dtale/main/1

D-Tale Features:
  - Interactive data exploration
  - Column statistics
  - Filtering & sorting
  - Visualization tools

Note: Keep this notebook running to access D-Tale.

To stop D-Tale, uncomment and run the cleanup cell below.


## 13. Execution Summary

In [19]:
end_time = datetime.now()
execution_time = (end_time - start_time).total_seconds()

print('\n' + '='*80)
print('EXECUTION COMPLETE')
print('='*80)

print(f'\n✓ EDA Pipeline finished successfully!')
print(f'  Execution Time: {execution_time:.2f} seconds')
print(f'  Output Directory: {dataset_output_dir}')

# List generated files
print(f'\nGenerated Reports:')
for item in os.listdir(dataset_output_dir):
    item_path = os.path.join(dataset_output_dir, item)
    if os.path.isfile(item_path):
        size_mb = os.path.getsize(item_path) / 1024**2
        print(f'  ✓ {item} ({size_mb:.2f} MB)')
    elif os.path.isdir(item_path):
        file_count = len([f for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f))])
        print(f'  ✓ {item}/ ({file_count} files)')

print(f'\nOpen these reports in a web browser to explore the data!')


EXECUTION COMPLETE

✓ EDA Pipeline finished successfully!
  Execution Time: 497.04 seconds
  Output Directory: reports/cleaned_salary_survey

Generated Reports:
  ✓ autoviz_plots/ (14 files)
  ✓ data_profile_report.html (0.01 MB)
  ✓ sweetviz_report.html (1.45 MB)

Open these reports in a web browser to explore the data!


## 14. Optional Cleanup

In [ ]:
# Uncomment to stop D-Tale
# if 'dtale_instance' in locals():
#     dtale_instance.kill()
#     print('D-Tale dashboard stopped.')